In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline




2026-05-30 14:30:10.097106: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-30 14:30:10.607576: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-30 14:30:13.117868: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Langkah 2: Import Dataset

In [3]:
dataset = pd.read_csv('weatherAUS.csv')
pd.unique(dataset['RainTomorrow'])

array(['No', 'Yes', nan], dtype=object)

In [4]:
print("Distribusi Kelas Asli :\n", dataset['RainTomorrow'].value_counts())


Distribusi Kelas Asli :
 RainTomorrow
No     110316
Yes     31877
Name: count, dtype: int64


# Data Quality Audit


In [5]:
missing_report = pd.DataFrame({
    "missing_count" : dataset.isnull().sum(),
    "missing_percent": dataset.isnull().mean() * 100
}).sort_values(by="missing_percent", ascending=False)

missing_report

,missing_count,missing_percent
Sunshine,69835,48.009762
Evaporation,62790,43.166506
Cloud3pm,59358,40.807095
Cloud9am,55888,38.421559
Pressure9am,15065,10.356799
Pressure3pm,15028,10.331363
WindDir9am,10566,7.263853
WindGustDir,10326,7.098859
WindGustSpeed,10263,7.055548
Humidity3pm,4507,3.098446


In [6]:
dataset = dataset.dropna(subset=['RainTomorrow'])
dataset['Date'] = pd.to_datetime(dataset['Date'], errors='coerce')
dataset = dataset.dropna(subset=['Date'])

# Langkah 3: Pisahkan Fitur dan Target

In [7]:
x = dataset.drop(columns=['RainTomorrow'])
y = dataset['RainTomorrow']




# Identifikasi kolom numerik dan kategorial

In [8]:
numeric_cols = x.select_dtypes(include=['number']).columns
categorial_cols = x.select_dtypes(include=['object']).columns


# Membuat Pipeline

In [ ]:
# 
numeric_pipeline = Pipeline([
    ("iterative", IterativeImputer(random_state=42, initial_strategy='median')),
    ("scaler", StandardScaler())
])

categorial_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent'))
])

preprocessing_pipline = Pipeline([
    ("numeric_cols", numeric_cols, numeric_pipeline),
    ("categorial_cols", categorial_cols, categorial_pipeline)
])

# Langkah 4: Melakukan Imputer & One Hot encode

# Imputer

In [ ]:


numeric_imputer = IterativeImputer(random_state=42, initial_strategy='median')
categorial_imputer = SimpleImputer(strategy='most_frequent')

In [10]:
dataset[numeric_cols] = numeric_imputer.fit_transform(dataset[numeric_cols])
dataset[categorial_cols] = categorial_imputer.fit_transform(dataset[categorial_cols])

/home/muhammad/MachineLearning/venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


# Encode Categorial 


In [23]:
# encoding fitur categorial

column_transfer = ColumnTransformer(transformers=[('encoder', OneHotEncoder(sparse_output=False, drop='first'), categorial_cols)], remainder='passthrough')

x_encode = column_transfer.fit_transform(x)
print(x_encode)

[[0.0 1.0 0.0 ... nan 16.9 21.8]
 [0.0 1.0 0.0 ... nan 17.2 24.3]
 [0.0 1.0 0.0 ... 2.0 21.0 23.2]
 ...
 [0.0 0.0 0.0 ... nan 10.9 24.5]
 [0.0 0.0 0.0 ... nan 12.5 26.1]
 [0.0 0.0 0.0 ... 2.0 15.1 26.0]]


# Encode Label

In [12]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_encode = label_encoder.fit_transform(y)




# Langkah 5: Feature Scalling

In [13]:
scale = StandardScaler()
dataset[numeric_cols] = scale.fit_transform(dataset[numeric_cols])

# Langkah 6: Split atau Stratified Split

# Temporal Split karenda datset imbalance

In [18]:
# Menggunakan temporal split untuk algoritma lstm(long short term memory)
split_index = int(len(dataset) * 0.8)
x_train = x.iloc[:split_index]
x_test = x.iloc[split_index:]

y_train = y_encode[:split_index]
y_test = y_encode[split_index:]


print("Jumlah data train : \n", len(x_train))
print("Jumlah data train : \n", len(x_test))


Jumlah data train : 
 113754
Jumlah data train : 
 28439


# Menggunakan Algoritma klasifikasi biasa

In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x, y_encode, test_size=0.2, random_state=42, stratify=y_encode)


print("Jumlah data train : \n", len(x_train))
print("Jumlah data train : \n", len(x_test))

Jumlah data train : 
 113754
Jumlah data train : 
 28439


1. Data Quality Audit: Identifikasi persentase missing values pada tiap fitur dan melakukan
pembersihan baris yang tidak valid.
2. Smart Imputation: Menerapkan strategi imputasi (Mean/Median/Mode atau Iterative Imputer)
untuk mengisi data cuaca yang hilang.

3. Categorical Encoding: Transformasi fitur lokasi dan arah angin menjadi representasi numerik (One-
Hot atau Target Encoding).

4. Feature Scalling: Melakukan standarisasi data menggunakan StandardScaler untuk variabel atmosfer
yang memiliki rentang berbeda.